# Module 2: Silver layer, cleaning and standardising

**Input:** `workspace.dubai_re.bronze_transactions` (1,814,844 rows, all text)
**Output:** `workspace.dubai_re.silver_transactions` (same rows, proper types, cleaned values, new analysis columns)
**Time:** 2–3 hours, so split it over two sittings if you like. Every step builds on the previous DataFrame, so after a break re-run from the top.

What you'll practise: renaming with a mapping, safe type casting, fixing bad values, `when/otherwise` logic, regular expressions, approximate percentiles, joins, and automated data quality tests.

Same rules as Module 1: replace every `___`, run the ✅ checks, and only open `solutions/02_silver_clean_solution.ipynb` after a real attempt.

## 🧠 Why we're doing this: what "silver" means

Silver is where you turn *what the source sent* into *what the business means*. Every decision you make here is written down in code, so anyone can see exactly how a raw value became a number in the dashboard.

The rules for this project come straight from the data quality findings in `docs/data_profile.md`:

| Finding | Silver rule |
|---------|-------------|
| DQ1 Dates are `YYYY-MM-DD` | Parse with an explicit format |
| DQ2 Four Hijri-year dates | Flag `is_valid_date = false` |
| DQ5 Arabic `أخرى` in an English column | Map to `Other` |
| DQ6 Lease-to-own recorded twice | `is_core_sale` keeps only true sales |
| DQ8 DLD land names vs market names | New `community` column |
| DQ9 `rooms_en` mixes bedrooms and labels | New `bedrooms` number |
| DQ10 Extreme prices | Flag with `is_price_outlier` (flag, don't delete) |
| DQ12 Rent columns aren't rent | Drop them |

**Silver keeps every row.** We *flag* problems instead of deleting rows, so each analysis can choose its own filter and the totals still reconcile with bronze.

🎤 **Interview angle:** *"How do you handle bad data?"* → Profile first, write each rule down, flag rather than delete where possible, and test the output automatically before anyone uses it.

## Setup

In [ ]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA  = "dubai_re"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_transactions"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_transactions"

def check(label, actual, expected):
    fmt = (lambda v: f"{v:,}") if isinstance(expected, int) else str
    if actual == expected:
        print(f"✅ {label}: {fmt(actual)}")
    else:
        print(f"❌ {label}: got {actual}, expected {fmt(expected)}. Re-read the hint above this cell.")

def check_close(label, actual, expected, tolerance=0.02):
    """For approximate results: passes if within ±tolerance (2% by default)."""
    if actual is None:
        print(f"❌ {label}: got nothing back (None). Check the cell above.")
        return
    ok = abs(actual - expected) <= tolerance * expected
    print(f"{'✅' if ok else '❌'} {label}: got {actual:,.0f}, expected about {expected:,.0f} (±{tolerance:.0%})")

## Step 1: Read the bronze table

🧠 **Why we're doing this**
In Module 1 you read files with `spark.read`. Now the data is a registered table, so you read it by name with `spark.table("catalog.schema.table")`. No paths, no file formats, no options. This is one of the main benefits of saving to Unity Catalog.

✏️ **TODO 1:** read `BRONZE_TABLE` into `df_bronze`.

In [ ]:
# TODO 1: read the bronze table
df_bronze = spark.table(BRONZE_TABLE)

check("Bronze rows", df_bronze.count(), 1_814_844)

## Step 2: Keep the columns we need, with clear names

🧠 **Why we're doing this**
Bronze has 49 columns: 14 are Arabic copies, 5 are ID codes that duplicate a text column, and 3 we decided to drop (`rent_value`, `meter_rent_price`, `load_timestamp`). Names like `actual_worth` and `procedure_area` make every later query harder to read.

So we keep 27 columns and give them business-friendly `snake_case` names. The mapping is a Python **dictionary**: `{old_name: new_name}`. It works as a mini data dictionary and lives in code, where it can't drift out of date.

**List comprehension** is Python's one-line loop for building a list:
```python
[F.col(old).alias(new) for old, new in RENAME.items()]
```
reads as *"for every (old, new) pair, make the column `old` renamed to `new`"*. `.alias()` is `AS` in SQL. Passing that list to `.select()` is the same as writing a 27-line `SELECT old AS new, …`.

✏️ **TODO 2:** complete the list comprehension inside `.select(...)`.

In [ ]:
RENAME = {
    # identity and transaction
    "transaction_id":       "transaction_id",
    "instance_date":        "transaction_date",
    "trans_group_en":       "transaction_group",
    "procedure_name_en":    "procedure",
    "reg_type_en":          "registration_type",
    # property
    "property_type_en":     "property_type",
    "property_sub_type_en": "property_sub_type",
    "property_usage_en":    "property_usage",
    "rooms_en":             "rooms_raw",
    "has_parking":          "has_parking",
    # money and size
    "actual_worth":         "price_aed",
    "procedure_area":       "area_sqm",
    "meter_sale_price":     "price_per_sqm",
    # location
    "area_id":              "area_id",
    "area_name_en":         "dld_area",
    "master_project_en":    "master_project",
    "project_number":       "project_number",
    "project_name_en":      "project",
    "building_name_en":     "building",
    "nearest_metro_en":     "nearest_metro",
    "nearest_mall_en":      "nearest_mall",
    "nearest_landmark_en":  "nearest_landmark",
    # parties
    "no_of_parties_role_1": "parties_role_1",
    "no_of_parties_role_2": "parties_role_2",
    "no_of_parties_role_3": "parties_role_3",
    # lineage from Module 1
    "_source_file":         "_source_file",
    "_ingested_at":         "_ingested_at",
}

# TODO 2: select every column in RENAME under its new name
df_renamed = df_bronze.select([F.col(old).alias(new) for old, new in RENAME.items()])

check("Columns kept", len(df_renamed.columns), 27)
print(df_renamed.columns)

## Step 3: Check for "nulls in disguise"

🧠 **Why we're doing this**
A blank can be stored as a real `NULL` or as an empty string `""`. They behave differently: `COUNT(col)` skips NULLs but counts `""`, and `COALESCE` only replaces NULLs. Before casting, confirm which one you have.

Spark's CSV reader turned the source's `""` into real NULLs, so this check should find **zero** empty strings. Nothing to fix, but now you *know* rather than assume. This cell is written for you. Read how it builds one count per text column in a single pass.

In [ ]:
text_cols = [c for c, t in df_renamed.dtypes if t == "string"]

empty_counts = df_renamed.select([
    F.sum((F.trim(F.col(c)) == "").cast("int")).alias(c) for c in text_cols
]).first().asDict()

total_empty = sum(v or 0 for v in empty_counts.values())
check("Empty-string values across all text columns", total_empty, 0)

## Step 4: Give every column its proper type

🧠 **Why we're doing this**
In bronze everything is text, so `"980000.00" > "1350000.00"` is *true*, because text compares character by character. You can't sum, average or sort numbers stored as text.

**ANSI mode and `try_cast`.** Databricks runs Spark in **ANSI mode**: a normal `CAST("abc" AS DOUBLE)` *fails the whole job* instead of quietly returning NULL. That's good for catching surprises, but one bad value in 1.8M rows would stop everything. `try_cast` returns NULL for values it can't convert, and then **we count the NULLs** to see whether anything failed.

| Column(s) | From text | To | How |
|-----------|-----------|----|-----|
| price, area, price per sq m | `"980000.00"` | `DOUBLE` | `try_cast(… AS DOUBLE)` |
| area_id, parties | `"435"` | `INT` | `try_cast(… AS INT)` |
| project_number | `"1960.00"` | `INT` | double first, then int |
| transaction_date | `"2022-07-19"` | `DATE` | `F.to_date(col, "yyyy-MM-dd")` (DQ1) |
| has_parking | `"1"` / `"0"` | `BOOLEAN` | `col == "1"` |

`F.expr("…")` lets you write a piece of SQL inside Python. Handy when SQL has the neater syntax.

✏️ **TODO 4a:** inside the loops, cast each column with `F.expr(f"try_cast({c} AS DOUBLE)")` (or `INT`).
✏️ **TODO 4b:** parse the date with `F.to_date("transaction_date", "yyyy-MM-dd")`.

In [ ]:
DOUBLE_COLS = ["price_aed", "area_sqm", "price_per_sqm"]
INT_COLS    = ["area_id", "parties_role_1", "parties_role_2", "parties_role_3"]

df_typed = df_renamed

# TODO 4a: cast the numeric columns
for c in DOUBLE_COLS:
    df_typed = df_typed.withColumn(c, F.expr(f"try_cast({c} AS DOUBLE)"))
for c in INT_COLS:
    df_typed = df_typed.withColumn(c, F.expr(f"try_cast({c} AS INT)"))

# TODO 4b: parse the date
df_typed = (
    df_typed
      .withColumn("transaction_date", F.to_date("transaction_date", "yyyy-MM-dd"))
      .withColumn("project_number", F.expr("CAST(try_cast(project_number AS DOUBLE) AS INT)"))
      .withColumn("has_parking", F.col("has_parking") == "1")
)

df_typed.printSchema()

Now prove the casting didn't silently lose anything. Price, area and date must have **zero** NULLs. `parties_role_1` has 973 NULLs, which were already blank in the source (0.1%), so that's expected, not a casting failure.

In [ ]:
nulls = df_typed.select(
    F.sum(F.col("price_aed").isNull().cast("int")).alias("price_aed"),
    F.sum(F.col("area_sqm").isNull().cast("int")).alias("area_sqm"),
    F.sum(F.col("transaction_date").isNull().cast("int")).alias("transaction_date"),
    F.sum(F.col("parties_role_1").isNull().cast("int")).alias("parties_role_1"),
).first()

check("NULL prices after cast", nulls["price_aed"], 0)
check("NULL areas after cast", nulls["area_sqm"], 0)
check("NULL dates after parse", nulls["transaction_date"], 0)
check("NULL parties_role_1 (blank in source)", nulls["parties_role_1"], 973)

## Step 5: Fix known bad values

🧠 **Why we're doing this**
`F.when(condition, value).otherwise(other)` is SQL's `CASE WHEN … THEN … ELSE … END` and Excel's `IF()`. You'll use it more than almost any other function.

Two fixes from the profile:
- **DQ5:** `property_usage` contains the Arabic word `أخرى` ("Other") in 6,348 rows. Map it to `"Other"`.
- **DQ2:** four rows have years like 1416 (Hijri years typed as Gregorian). Flag them with `is_valid_date`. We *don't* delete them, so the totals still match bronze.

✏️ **TODO 5:** complete the `when` for `property_usage`: when it equals `"أخرى"`, use `"Other"`, otherwise keep the existing value.
*Hint:* `.otherwise(F.col("property_usage"))`

In [ ]:
# TODO 5: fix property_usage and flag impossible dates
df_fixed = (
    df_typed
      .withColumn("property_usage",
                  F.when(F.col("property_usage") == "أخرى", "Other")
                   .otherwise(F.col("property_usage")))
      .withColumn("is_valid_date", F.year("transaction_date") >= 1960)
)

usage_counts = {r["property_usage"]: r["count"] for r in df_fixed.groupBy("property_usage").count().collect()}
check("Distinct property_usage values", len(usage_counts), 9)
check("Rows now labelled Other", usage_counts.get("Other"), 55_451)
check("Invalid dates flagged", df_fixed.filter(~F.col("is_valid_date")).count(), 4)

## Step 6: Add the analysis columns

🧠 **Why we're doing this**
These are the columns every later module needs. Build them once here, so each rule has exactly one definition.

| New column | Rule | Why |
|------------|------|-----|
| `community` | `master_project`, or `dld_area` if blank | Market names people search for (DQ8). `F.coalesce` = first non-null, the same as SQL's `COALESCE` |
| `bedrooms` | `Studio` → 0, `"2 B/R"` → 2, anything else → NULL | DQ9. Uses a **regular expression** (below) |
| `area_sqft` | `area_sqm × 10.7639` | Dubai prices are quoted per sq ft |
| `price_per_sqft` | `price_aed / area_sqft`, only when area > 0 | In ANSI mode, dividing by zero is an error, so guard it |
| `is_off_plan` | `registration_type == "Off-Plan Properties"` | Q2 |
| `is_core_sale` | `procedure` in Sell, Sell - Pre registration, Delayed Sell | Real market sales only (DQ6) |
| `transaction_month` | First day of the month | Monthly trends and the Q5 forecast |

**Regular expressions in 30 seconds.** `^\d+ B/R$` means: start of text (`^`), one or more digits (`\d+`), a space, `B/R`, end of text (`$`). So it matches `"2 B/R"` but not `"PENTHOUSE"`. `F.regexp_extract("rooms_raw", r"^(\d+)", 1)` pulls out the digits inside the brackets (group 1). The `r"..."` prefix makes it a *raw string*, so Python leaves the backslashes alone.

✏️ **TODO 6a:** `community`: coalesce `master_project` and `dld_area`.
✏️ **TODO 6b:** `is_core_sale`: `F.col("procedure").isin(CORE_SALE_PROCEDURES)`. `.isin()` is SQL's `IN (…)`.

In [ ]:
CORE_SALE_PROCEDURES = ["Sell", "Sell - Pre registration", "Delayed Sell"]
SQFT_PER_SQM = 10.7639

df_enriched = (
    df_fixed
      # TODO 6a
      .withColumn("community", F.coalesce("master_project", "dld_area"))
      .withColumn("bedrooms",
                  F.when(F.col("rooms_raw") == "Studio", 0)
                   .when(F.col("rooms_raw").rlike(r"^\d+ B/R$"),
                         F.regexp_extract("rooms_raw", r"^(\d+)", 1).cast("int")))
      .withColumn("area_sqft", F.col("area_sqm") * SQFT_PER_SQM)
      .withColumn("price_per_sqft",
                  F.when(F.col("area_sqft") > 0, F.col("price_aed") / F.col("area_sqft")))
      .withColumn("is_off_plan", F.col("registration_type") == "Off-Plan Properties")
      # TODO 6b
      .withColumn("is_core_sale", F.col("procedure").isin(CORE_SALE_PROCEDURES))
      .withColumn("transaction_month", F.trunc("transaction_date", "month"))
)

core = df_enriched.filter("is_core_sale")
beds = {r["bedrooms"]: r["count"] for r in core.groupBy("bedrooms").count().collect()}

check("Core sales", core.count(), 1_314_686)
check("Core sales: studios", beds.get(0), 232_351)
check("Core sales: 1-bed", beds.get(1), 391_819)
check("Core sales: communities", core.select("community").distinct().count(), 373)
check("Rows with no community", df_enriched.filter(F.col("community").isNull()).count(), 0)

🤔 **Think about it:**
1. Why did we make `bedrooms` NULL for offices and shops, instead of 0?
2. `is_core_sale` is false for mortgages, gifts *and* lease-to-own sales. Why exclude lease-to-own from price analysis? (Hint: DQ6.)

**Answers**
1. 0 means *studio*, a real value. An office has no bedroom count at all, so NULL ("not applicable") is honest. Averages and counts then ignore offices instead of treating them as studios.
2. Each lease-to-own deal appears twice (a Sales leg and a Mortgages leg with linked IDs), and its value is a financing amount, not an open-market price. Including it would double count and distort prices.

## Step 7: Flag price outliers

🧠 **Why we're doing this**
A few registrations have prices no market buyer would pay (AED 46 per sq ft for a flat, or AED 660M for one unit). Usually they're data entry errors, internal transfers or bulk deals. They'd wreck averages. Medians resist them, but they still distort rankings in small communities.

**The rule:** a core sale is an outlier if its price per sq ft is below the **0.5th** or above the **99.5th percentile** *for its property type* (units, villas, land and buildings have very different price levels). That flags about 1% of sales. We **flag, not delete**, so Module 4 can show results with and without them.

**Three new techniques:**
1. **`groupBy().agg()`**: SQL's `GROUP BY`. One row of percentiles per property type.
2. **`F.percentile_approx`**: an *exact* percentile has to sort all 1.3M values. The approximate version is far cheaper, and accurate to within a tiny fraction of a percentile. That's why the check below allows ±2%.
3. **Join + `F.broadcast`**: attach the 4-row bounds table to every transaction. `broadcast` tells Spark to send the small table to every worker instead of shuffling the big one, which is the standard trick for "big table joined to small lookup".

✏️ **TODO 7a:** compute the 0.995 percentile next to the 0.005 one.
✏️ **TODO 7b:** the outlier condition: price per sq ft `<` the low bound **or** (`|`) `>` the high bound. In PySpark, wrap each comparison in brackets.

In [ ]:
# TODO 7a: percentile bounds per property type, from core sales only
bounds = (
    df_enriched
      .filter("is_core_sale")
      .groupBy("property_type")
      .agg(F.percentile_approx("price_per_sqft", 0.005).alias("ppsf_low"),
           F.percentile_approx("price_per_sqft", 0.995).alias("ppsf_high"))
)
display(bounds)

# TODO 7b: flag outliers (only meaningful for core sales; NULL for everything else)
df_silver = (
    df_enriched
      .join(F.broadcast(bounds), on="property_type", how="left")
      .withColumn("is_price_outlier",
                  F.when(F.col("is_core_sale"),
                         (F.col("price_per_sqft") < F.col("ppsf_low")) | (F.col("price_per_sqft") > F.col("ppsf_high"))))
      .drop("ppsf_low", "ppsf_high")
)

check("Rows after join (a join must never add rows here)", df_silver.count(), 1_814_844)
check_close("Core sales flagged as outliers", df_silver.filter("is_price_outlier").count(), 13_133)

💡 **Why check the row count after a join?** If the lookup table had *two* rows for `Unit`, every unit sale would be duplicated. That's called *join fan-out*, and it's one of the most common causes of inflated totals in real dashboards. Checking the count after every join is a habit worth showing in interviews.

## Step 8: Save the silver table

🧠 **Why we're doing this**
Same pattern as bronze: Delta, overwrite, so it's idempotent. We also put the columns in a sensible order, because people browsing the table in Catalog Explorer (or Power BI) read left to right.

**Partitioning?** You may read that big tables should be partitioned (e.g. by year). At 1.8M rows this table is small for Spark. Partitioning it would create many tiny files and make queries *slower*. A common rule of thumb: don't partition tables under about 1 TB. Knowing when *not* to apply a technique is a good interview answer.

✏️ **TODO 8:** write `df_silver` to `SILVER_TABLE` (delta, overwrite).

In [ ]:
SILVER_COLUMNS = [
    "transaction_id", "transaction_date", "transaction_month", "is_valid_date",
    "transaction_group", "procedure", "is_core_sale", "registration_type", "is_off_plan",
    "property_type", "property_sub_type", "property_usage", "rooms_raw", "bedrooms", "has_parking",
    "price_aed", "area_sqm", "area_sqft", "price_per_sqm", "price_per_sqft", "is_price_outlier",
    "community", "master_project", "dld_area", "area_id", "project", "project_number", "building",
    "nearest_metro", "nearest_mall", "nearest_landmark",
    "parties_role_1", "parties_role_2", "parties_role_3",
    "_source_file", "_ingested_at",
]

# TODO 8: save as a Delta table
(
    df_silver.select(SILVER_COLUMNS)
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(SILVER_TABLE)
)

check("Silver columns", len(spark.table(SILVER_TABLE).columns), 36)

Add a description, so anyone browsing the catalog knows what this table is. Just run it.

In [ ]:
%sql
COMMENT ON TABLE workspace.dubai_re.silver_transactions IS
'Cleaned DLD real estate transactions: one row per registered transaction (all groups). Typed columns, community and bedrooms standardised, is_core_sale and is_price_outlier flags. Source: bronze_transactions. Rules: docs/data_profile.md.'

## Step 9: Data quality gate

🧠 **Why we're doing this**
Every check so far was *you* eyeballing a ✅. A **data quality gate** is a set of tests that runs automatically and **stops the pipeline** if any fail, so a bad table never reaches a dashboard. In industry, tools like dbt tests, Great Expectations or Databricks' DLT expectations do this. The idea is identical to what you're writing here: a list of yes/no rules.

We test the saved table (reading Delta is fast), and `raise` an error if anything fails. That would also fail a scheduled job.

✏️ **TODO 9:** add the uniqueness test: the number of distinct `transaction_id` values must equal 1,814,844.

In [ ]:
s = spark.table(SILVER_TABLE)

tests = {
    "row count matches bronze":            s.count() == 1_814_844,
    # TODO 9: transaction_id is unique
    "transaction_id is unique":            s.select("transaction_id").distinct().count() == 1_814_844,
    "no NULL transaction dates":           s.filter(F.col("transaction_date").isNull()).count() == 0,
    "no NULL prices":                      s.filter(F.col("price_aed").isNull()).count() == 0,
    "no Arabic text in property_usage":    s.filter(F.col("property_usage") == "أخرى").count() == 0,
    "core sales have price_per_sqft > 0":  s.filter("is_core_sale AND (price_per_sqft IS NULL OR price_per_sqft <= 0)").count() == 0,
    "outlier flag only on core sales":     s.filter("NOT is_core_sale AND is_price_outlier IS NOT NULL").count() == 0,
    "bedrooms between 0 and 10":           s.filter("bedrooms < 0 OR bedrooms > 10").count() == 0,
}

for name, passed in tests.items():
    print(f"{'✅' if passed else '❌'} {name}")

failed = [name for name, passed in tests.items() if not passed]
if failed:
    raise ValueError(f"Data quality gate failed: {failed}")
print("\n🎉 Silver passed all data quality tests.")

## Step 10: First real question in SQL

🧠 **Why we're doing this**
Now the payoff: with clean types and flags, a market question is a short, readable query. This is a first look at **Q1**: median AED per sq ft for residential flats, by year.

`MEDIAN()` is preferred to `AVG()` for prices: one AED 50M penthouse moves an average a lot but barely moves a median.

✏️ **TODO 10:** fill in the `WHERE` clause so it keeps only:
- core sales (`is_core_sale`)
- that are **not** outliers
- with `property_sub_type = 'Flat'` and `property_usage = 'Residential'`
- from 2015 onwards

In [ ]:
%sql
-- TODO 10: median price per sq ft for residential flats, per year
SELECT YEAR(transaction_date)          AS year,
       COUNT(*)                        AS flat_sales,
       ROUND(AVG(CAST(is_off_plan AS INT)) * 100, 1) AS off_plan_pct,
       ROUND(MEDIAN(price_per_sqft))   AS median_aed_per_sqft
FROM workspace.dubai_re.silver_transactions
WHERE is_core_sale
  AND NOT is_price_outlier
  AND property_sub_type = 'Flat'
  AND property_usage = 'Residential'
  AND transaction_date >= '2015-01-01'
GROUP BY YEAR(transaction_date)
ORDER BY year

In [ ]:
median_2025 = spark.sql(f"""
    SELECT MEDIAN(price_per_sqft) AS m
    FROM {SILVER_TABLE}
    WHERE is_core_sale AND NOT is_price_outlier
      AND property_sub_type = 'Flat' AND property_usage = 'Residential'
      AND YEAR(transaction_date) = 2025
""").first()["m"]

check_close("2025 median AED per sq ft (flats)", median_2025, 1_734, tolerance=0.01)

💡 Click the **+** next to the results table, choose **Visualization → Line**, and put `year` on the X axis and `median_aed_per_sqft` on the Y axis. You've just drawn the first chart of the project. Look at 2020 (COVID) and the climb from 2021.

## ✅ Module 2 complete

**What you built:** `silver_transactions`, 1,814,844 rows and 36 typed columns, with every cleaning rule written in code and checked by an automatic quality gate.

**🎤 Practise saying these out loud** (model answers in `LEARNING.md`):
1. Why use `try_cast` instead of `cast`, and what is ANSI mode?
2. Why flag outliers instead of deleting them? How did you define an outlier?
3. What is join fan-out and how do you guard against it?
4. Why `percentile_approx` rather than an exact percentile?
5. What's a data quality gate, and what did yours test?
6. Why didn't you partition the table?

**Commit your work**
1. **File → Export → IPython Notebook**, then save over `databricks/02_silver_clean.ipynb`
2. In the VS Code terminal:
```bash
git add databricks/02_silver_clean.ipynb
git commit -m "Add silver layer: typed and standardised transactions with DQ gate"
git push
```
3. Tick Module 2 in the README roadmap.

**Next:** Module 3, Gold. A star schema (fact table plus date, location and property dimensions) and the aggregate tables that Q1, Q2 and Q5 run on.